# CyberVision Gen — Personne 1 : données et qualité
**Mission** : données propres, protocole sans fuite. Ce notebook exécute le pipeline `cvgen_data` **étape par étape**.
Chaque étape affiche son rapport et se termine par des **vérifications** (`✅`) : si une vérification échoue, on s'arrête.

Ordre (imposé par la fiche 11 et le protocole commun) :
1. Chargement · 2. Nettoyage · 3. Doublons exacts **avant** partition + classes · 4. Partition temporelle intra-fenêtre ·
5. Sélection des variables (train) + doublons résiduels · 6. Schéma & contraintes · 7. Normalisation (train) ·
8. Matrices 8×8 · 9. Sauvegarde des partitions figées + régime réduit · 10. Audit des quasi-doublons · 11. Data card

## 0. Installation (Colab)
Option A : dépôt GitHub de l'équipe. Option B : zip `cybervision_data.zip` déposé sur Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Option A
# !git clone https://github.com/<equipe>/cybervision-gen.git /content/cybervision-gen
# %cd /content/cybervision-gen/cybervision_data

# Option B
!unzip -oq /content/drive/MyDrive/CyberVision/cybervision_data.zip -d /content/
%cd /content/cybervision_data
!pip install -q -r requirements.txt

## Données brutes
Télécharger CIC-IDS2017 depuis https://www.unb.ca/cic/datasets/ids-2017.html (formulaire) :
**`GeneratedLabelledFlows.zip`** (dossier *TrafficLabelling*, contient `Timestamp`) — c'est la version requise pour la partition temporelle.
`MachineLearningCSV.zip` n'a **pas** de Timestamp : le pipeline fonctionne mais se replie sur l'ordre des lignes (à déclarer comme limite).
Version corrigée (recommandée si le temps le permet) : WTMC-2021, Engelen et al. — https://intrusion-detection.distrinet-research.be/WTMC2021/tools_datasets.html → `flavor: wtmc2021`.

Placer les 8 CSV dans `data/raw/` (ou pointer `paths.raw_dir` vers Drive). **Ne jamais committer les CSV.**

In [ ]:
import os, glob
RAW = '/content/drive/MyDrive/CyberVision/raw'   # <- adapter
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/raw'):
    os.symlink(RAW, 'data/raw')
sorted(glob.glob('data/raw/*.csv'))

In [ ]:
 %pip install pandas numpy scikit-learn matplotlib pyarrow pyyaml joblib tabulate pytest ipykernel

In [ ]:
import os, sys
os.chdir(r"C:\Users\hp\Downloads\CYBERVISION-GEN\cybervision-gen\cybervision_data")
sys.path.insert(0, os.getcwd())
print(os.getcwd())
print(os.listdir())

In [ ]:
import json, logging, pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
from cvgen_data import DataPipeline, load_config
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', force=True)
pd.set_option('display.max_rows', 100); pd.set_option('display.width', 200)

cfg = load_config('config.yaml')
p = DataPipeline(cfg)
show = lambda d: print(json.dumps(d, indent=2, default=str, ensure_ascii=False))
fig = lambda n: display(Image(f"{p.fig_dir}/{n}"))

## 1. Chargement
À vérifier : `fix_12h_applied` (heures d'après-midi écrites 1:00–5:59), `n_unparsed` = 0, `spearman_row_vs_time` proche de 1, labels sans caractère illisible.

In [ ]:
rep = p.step1_load()
pd.DataFrame({f: {k: v for k, v in r.items() if k != 'labels'} for f, r in rep['files'].items()}).T

In [ ]:
print('Colonnes normalisées :', len(rep['columns']))
print(rep['columns'])
pd.DataFrame({f: r['labels'] for f, r in rep['files'].items()}).fillna(0).astype(int)

In [ ]:
# validations
assert rep['has_timestamp'] or cfg['source']['flavor'] == 'ml_cve', "Pas de Timestamp : utiliser TrafficLabelling"
assert all(r.get('n_unparsed', 0) == 0 for r in rep['files'].values()), "timestamps illisibles"
assert not any('\ufffd' in l or '\x96' in l for l in p.df['label_orig'].unique())
print('chargement OK —', len(p.df), 'lignes')

## 2. Nettoyage (inf, NaN, négatifs, labels « Attempted »)

In [ ]:
rep = p.step2_clean(); show(rep)

## 3. Doublons exacts (avant partition) et classes
Conflits = même vecteur, labels différents → supprimés. Doublons → 1re occurrence chronologique gardée.

In [ ]:
rep = p.step3_dedup_and_classes()
t = pd.DataFrame({'avant': rep['dedup']['per_class_before'], 'après': rep['dedup']['per_class_after'],
                  'doublons retirés': rep['dedup']['exact_duplicates_removed']['per_class'],
                  'conflits': rep['dedup']['conflicting_label_rows']['per_class']}).fillna(0).astype(int)
t['% retiré'] = (100 * (1 - t['après'] / t['avant'])).round(1)
display(t.sort_values('avant', ascending=False))
show(rep['classes']); fig('01_class_distribution.png')

## 4. Partition temporelle intra-fenêtre
Vérifié automatiquement : max(t train) ≤ min(t val) ≤ … dans chaque fenêtre, et chaque classe dans chaque split.

In [ ]:
rep = p.step4_split()
display(pd.DataFrame(rep['windows']))
display(pd.crosstab(p.df['label'], p.df['split'], margins=True)[['train', 'val', 'test', 'All']])
fig('02_split_timeline.png')

## 5. Sélection des variables (train uniquement) + doublons résiduels

In [ ]:
rep = p.step5_features()
show({k: rep[k] for k in ['n_candidates', 'n_selected', 'dropped_identifiers', 'dropped_constant',
                          'dropped_identical', 'dropped_correlation', 'post_selection_dedup', 'cross_split_overlap']})
print(p.feats)
fig('03_feature_histograms.png'); fig('04_correlations.png')

## 6. Schéma typé et contraintes de plausibilité
Livrable pour la **Personne 3** : `artifacts/feature_schema.json`. Une contrainte violée par > 1 % des flux **réels** est marquée `unreliable` — ne pas l'imposer au générateur.

In [ ]:
rep = p.step6_schema_constraints(); show(rep)
viol = pd.read_csv(f"{p.P['reports_dir']}/constraint_violations_real_train.csv", index_col=0)
viol[viol['global'] > 0].round(4)

In [ ]:
pd.DataFrame(p.schema['features'])[['name', 'type', 'integer', 'nonnegative', 'train_min', 'train_max']]

## 7. Normalisation (ajustée sur le train)
log1p signé puis min-max [-1, 1] (sortie `tanh` du générateur). Val/test ne sont pas écrêtés : la part hors plage mesure le décalage temporel.

In [ ]:
rep = p.step7_scale(); show(rep)

## 8. Matrices 8×8 + masque

In [ ]:
rep = p.step8_matrix(); show(rep)
print(np.array(p.layout.mask, dtype=int)); fig('05_matrices_8x8.png')

## 9. Sauvegarde des partitions figées + régime réduit
À committer : `data/processed/splits/*.csv.gz`, `MANIFEST.json`, `artifacts/*.json`, `config.yaml`.

In [ ]:
rep = p.step9_save_splits()
display(pd.DataFrame(rep['reduced_regime_counts']).fillna(0).astype(int)); show(rep['manifest'])

## 10. Audit des quasi-doublons
τ par classe = 1er centile des distances val → train (seuil fixé sur la **validation**). Réutilisé par la Personne 4 sur le synthétique.

In [ ]:
rep = p.step10_audit()
display(pd.DataFrame({c: {'tau': v['tau'], 'med val→train': v['val_to_train']['median'],
                          'med test→train': v['test_to_train']['median'],
                          'quasi-doublons test %': 100 * v['test_to_train']['near_dup_rate']} for c, v in rep.items()}).T.round(4))
fig('06_nn_distances.png')

## 11. Data card

In [ ]:
path = p.step11_datacard(); display(Markdown(open(path, encoding='utf-8').read()))

## 12. Tests de non-régression (sur données simulées)
À relancer après toute modification du code.

In [ ]:
!python -m pytest -q tests/

In [ ]:
import json, os
r = json.load(open("reports/pipeline_report.json", encoding="utf-8"))
print("Lignes chargées :", r["load"]["n_rows_total"])
for f, v in r["load"]["files"].items():
    print(f"  {f}: {v['rows_read']} lignes | 12h corrigé={v.get('fix_12h_applied')} | illisibles={v.get('n_unparsed')} | spearman={v.get('spearman_row_vs_time', 0):.3f}")
print("\nDoublons retirés :", r["dedup"]["exact_duplicates_removed"]["total"], "| conflits :", r["dedup"]["conflicting_label_rows"]["total"])
print("Lignes finales :", r["dedup"]["rows_after"])
print("\nEffectifs par split :")
print(pd.crosstab(p.df["label"], p.df["split"])[["train", "val", "test"]])
print("\nVariables retenues :", r["features"]["n_selected"], "| chevauchement :", r["features"]["cross_split_overlap"])
print("Contraintes non fiables :", r["constraints"]["unreliable"])
print("Erreur inverse scaler :", r["scaling"]["inverse_rel_error_max"])
print("\nFichiers créés :", os.listdir("data/processed"), os.listdir("artifacts"))

In [ ]:
print(pd.crosstab(p.df["label"], p.df["split"])[["train", "val", "test"]])
print("\nConflits par classe :", r["dedup"]["conflicting_label_rows"]["per_class"])
print("\nDoublons par classe :", r["dedup"]["exact_duplicates_removed"]["per_class"])
for f, v in r["load"]["files"].items():
    print(f, "| 12h :", v.get("fix_12h_applied"), "| illisibles :", v.get("n_unparsed"), "| spearman :", round(v.get("spearman_row_vs_time", 0), 3))
print("\nExclues :", r["classes"]["excluded_explicit"], r["classes"]["excluded_min_count"])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
tr = p.df[p.df.split == "train"].sample(300000, random_state=42)
va = p.df[p.df.split == "val"]
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42, class_weight="balanced")
rf.fit(tr[p.feats], tr["label"])
print(classification_report(va["label"], rf.predict(va[p.feats]), digits=3))